# Preparation

Run `pip install -r requirements.txt`, then run this notebook.

# Initial set up

I'm importing packages and defining some globals for the notebook here. I set a random seed for reproducibility.

> **AI usage disclaimer:** I've come to embrace AI-assisted development, the work presented in this notebook has been partially generated using LLMs, however, I've supervised the entire process and ultimately decided which parts to change, update, and customize. This disclaimer is an indicator of my commitment to ethical, responsible, and transparent use of AI.

In [ ]:
import json
import random
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageFont

from datasets import Dataset

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen3VLForConditionalGeneration,
)
from qwen_vl_utils import process_vision_info

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

from trl import SFTConfig, SFTTrainer

In [ ]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
OUTPUT_DIR = "qwen3-vl-8b-health-table-extraction-lora"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Overall description

Since we are dealing with complex (and variable) form layouts, we have to represent this data in a structured way. The most natural representation (with many benchmarks available as well) is with standard HTML tables, which allow us to make use of rowspan and colspan to merge cells and create fairly complex tables in a digital form.

**The HTML table** is wrapped in a JSON envelope that carries key/value fields (patient identifiers *redacted for training*, form date, form type):

**AI generated sample**

```json
{
  "form_type": "basic_metabolic_panel",
  "fields": {"specimen_date": "2026-03-14", "ordering_provider": "REDACTED"},
  "table_html": "<table><tr><th>Test</th><th colspan=\"2\">Reference Range</th><th>Result</th></tr>...</table>"
}
```

The model's job is: given the form image, emit exactly this JSON object as a string, while still getting HTML's structural expressiveness for the table body.

# Synthetic training data [AI generated helper]

In [ ]:
def _render_synthetic_lab_form(path: Path, seed: int) -> dict:
    """Render a crude 'lab panel' table into a high-resolution PNG and return
    the matching ground-truth record. This stands in for a real, scanned
    health-form image + a human- or vendor-labeled structured target.

    In production this function is replaced entirely: images come from your
    document ingestion pipeline (scanner/fax/EHR export), and targets come
    from human annotation (e.g., via Label Studio with an HTML-table export
    plugin) or from a trusted upstream structured source used to
    bootstrap weak labels.
    """
    rng = random.Random(seed)
    tests = [
        ("Sodium", "136", "145", "mmol/L"),
        ("Potassium", "3.5", "5.1", "mmol/L"),
        ("Chloride", "98", "107", "mmol/L"),
        ("Glucose", "70", "99", "mg/dL"),
        ("BUN", "7", "20", "mg/dL"),
        ("Creatinine", "0.6", "1.3", "mg/dL"),
    ]

    # High-resolution canvas: scanned health forms are frequently
    # 2000-3000px on the long edge. We deliberately render at high res so
    # the notebook exercises the same resolution regime as real scans.
    width, height = 1700, 2200
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/Library/Fonts/SnellRoundhand.ttc", 36)
        font_small = ImageFont.truetype("/Library/Fonts/SnellRoundhand.ttc", 28)
    except Exception:
        font = ImageFont.load_default()
        font_small = font

    draw.text((60, 60), "COMPREHENSIVE METABOLIC PANEL", fill="black", font=font)
    draw.text((60, 110), f"Specimen Date: 2026-0{rng.randint(1,9)}-{rng.randint(10,28)}", fill="black", font=font_small)

    top = 220
    row_h = 70
    col_x = [60, 500, 800, 1050, 1350]
    headers = ["Test", "Result", "Ref Low", "Ref High", "Unit"]
    for x, h in zip(col_x, headers):
        draw.text((x, top), h, fill="black", font=font)
    draw.line([(60, top + 50), (1640, top + 50)], fill="black", width=3)

    rows_html = []
    for i, (name, lo, hi, unit) in enumerate(tests):
        y = top + 70 + i * row_h
        result = str(round(rng.uniform(float(lo) * 0.85, float(hi) * 1.15), 1))
        row_vals = [name, result, lo, hi, unit]
        for x, v in zip(col_x, row_vals):
            draw.text((x, y), v, fill="black", font=font_small)
        rows_html.append(
            "<tr><td>{}</td><td>{}</td><td>{}</td><td>{}</td><td>{}</td></tr>".format(*row_vals)
        )

    img.save(path)

    table_html = (
        "<table>"
        "<tr><th>Test</th><th>Result</th><th>Ref Low</th><th>Ref High</th><th>Unit</th></tr>"
        + "".join(rows_html)
        + "</table>"
    )
    target = {
        "form_type": "basic_metabolic_panel",
        "fields": {"specimen_date": "REDACTED", "ordering_provider": "REDACTED"},
        "table_html": table_html,
    }
    return {"image": str(path), "target": target}


def build_synthetic_manifest(n_examples: int, out_dir: str) -> list[dict]:
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    manifest = []
    for i in range(n_examples):
        img_path = out_path / f"form_{i:04d}.png"
        record = _render_synthetic_lab_form(img_path, seed=i)
        manifest.append(record)
    return manifest


synthetic_manifest = build_synthetic_manifest(n_examples=24, out_dir="./synthetic_health_forms")
print(f"Built {len(synthetic_manifest)} synthetic (image, target) pairs.")
print(json.dumps(synthetic_manifest[0], indent=2)[:600], "...")


# Qwen3-VL chat template

Qwen3-VL is instruction-tuned and expects input as a **list of role-tagged messages**, where each message's `content` is itself a list of typed content blocks (`{"type": "text", "text": ...}` or `{"type": "image", "image": ...}`).

For training, **three** messages are constructed per example: System message (general instructions), user turn (image and brief description), system turn (processing output used as ground truth).

The training consists on masking the ground truth and comparing the output to compute loss.


In [ ]:
SYSTEM_PROMPT = (
    "You are a meticulous medical document structuring assistant. You are shown an image "
    "of a health form that contains one or more tables. Your job is to transcribe the form's "
    "table(s) and a small set of header fields into a single JSON object with exactly this shape:\n\n"
    '{"form_type": <string>, "fields": {"specimen_date": <string or null>, '
    '"ordering_provider": <string or null>}, "table_html": <string>}\n\n'
    "Rules:\n"
    "1. `table_html` must be a valid, minimal HTML <table> using <tr>/<th>/<td> and, when the form "
    "uses merged cells, `rowspan`/`colspan` attributes that faithfully reproduce the visual layout.\n"
    "2. Transcribe every visible row and column; do not summarize, reorder, or drop rows.\n"
    "3. If a field is not present on the form, use null rather than guessing.\n"
    "4. Output ONLY the JSON object. No markdown fences, no commentary.\n"
    "5. Never fabricate patient-identifying values; if a field is redacted or illegible, output null."
)

USER_INSTRUCTION = "Extract this form's structured data as JSON, per the schema and rules above."

# AI generated
def format_example(record: dict) -> list[dict]:
    """Convert one manifest record into the 3-turn chat-message structure
    Qwen3-VL's processor expects.

    `record["image"]` may be a filesystem path, a URL, or a PIL.Image — the
    `qwen_vl_utils.process_vision_info` helper accepts all three inside the
    `{"type": "image", "image": ...}` block, which is convenient: we don't
    have to eagerly load every image into memory just to build the message
    list.
    """
    target_str = json.dumps(record["target"], ensure_ascii=False)
    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": SYSTEM_PROMPT}],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": record["image"]},
                {"type": "text", "text": USER_INSTRUCTION},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": target_str}],
        },
    ]


formatted_examples = [format_example(r) for r in synthetic_manifest]
print(formatted_examples[0][1])   # the user turn
print()
print(formatted_examples[0][2])   # the assistant turn


# Visual tokens budget

To account for high resolution images and prevent hallucinations caused by a model incapable of reading the image content we set a processor that automatically scales the image resolution to the nearest ViT grid:

MIN_PIXELS <= H*W <= MAX_PIXELS


In [ ]:
MIN_PIXELS = 256 * 32 * 32
MAX_PIXELS = 1920 * 32 * 32

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
print(processor)


# Batch input + mask tokens for inference [AI generated]

Tokenize the *prompt-only* rendering (system + user turn, with the generation prompt opened) separately from the *full* rendering (system + user + assistant turn), and mask everything up to the length of the prompt-only tokenization.

In [ ]:
IMAGE_TOKEN_STRINGS = ["<|vision_start|>", "<|vision_end|>", "<|image_pad|>"]

def _image_token_ids(proc) -> list[int]:
    """Look these up dynamically from the tokenizer rather than hardcoding
    integer IDs, and warn loudly if any don't resolve. These particular
    special-token strings have been stable across the Qwen-VL family
    (Qwen2-VL, Qwen2.5-VL, Qwen3-VL all share this convention as of this
    writing), but "has been stable so far" is exactly the kind of claim this
    notebook has already been burned by once this migration (see the bbox
    coordinate convention change in Section 1B/4B) -- so we verify rather
    than silently trust it.
    """
    ids = [proc.tokenizer.convert_tokens_to_ids(t) for t in IMAGE_TOKEN_STRINGS]
    resolved = [i for i in ids if i is not None and i >= 0]
    if len(resolved) != len(IMAGE_TOKEN_STRINGS):
        print(
            "WARNING: not all image special tokens resolved to an id "
            f"({resolved} from {IMAGE_TOKEN_STRINGS}). Inspect "
            "proc.tokenizer.special_tokens_map / additional_special_tokens "
            "before trusting the collator's loss masking."
        )
    return resolved


IMAGE_TOKEN_IDS = _image_token_ids(processor)
print("Image-related special token ids:", IMAGE_TOKEN_IDS)


def collate_fn(examples: list[list[dict]]) -> dict[str, torch.Tensor]:
    """Custom multimodal collator for trl.SFTTrainer.

    `examples` is a list of chat-message lists (system/user/assistant), i.e.
    exactly what `format_example()` produces. Each call receives one
    mini-batch worth of examples.
    """
    full_texts, prompt_texts, image_inputs = [], [], []

    for messages in examples:
        # Full rendering: system + user + assistant -> this is the sequence
        # we actually train on.
        full_texts.append(
            processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        )
        # Prompt-only rendering: system + user, with the generation prompt
        # opened -> used purely to measure how many leading tokens to mask.
        # We render this instead of just diffing strings so that the exact
        # same special-token insertion logic the model was pretrained with
        # is used to compute the boundary.
        prompt_texts.append(
            processor.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
        )
        imgs, _ = process_vision_info(messages)
        image_inputs.append(imgs[0] if imgs else None)

    batch = processor(
        text=full_texts,
        images=image_inputs,
        return_tensors="pt",
        padding=True,
    )

    labels = batch["input_ids"].clone()

    # 1) Mask padding.
    labels[labels == processor.tokenizer.pad_token_id] = -100

    # 2) Mask image-placeholder / vision-boundary tokens -- these carry no
    #    linguistic content to predict.
    for image_token_id in IMAGE_TOKEN_IDS:
        labels[labels == image_token_id] = -100

    # 3) Mask the prompt portion (system + user turn) of every sequence, so
    #    loss is computed only on the assistant's structured-output tokens.
    #    We recompute the prompt length per-example with the SAME tokenizer
    #    call path (tokenize=True) so left-padding side / truncation cannot
    #    silently desync the boundary from the batch's actual input_ids.
    for i, prompt_text in enumerate(prompt_texts):
        prompt_len = len(processor.tokenizer(prompt_text, add_special_tokens=False).input_ids)
        labels[i, :prompt_len] = -100

    batch["labels"] = labels
    return batch
